In [ ]:
!pip install claude_agent_sdk

In [15]:
import json 
import os 
import shutil 
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd 
from claude_agent_sdk import ResultMessage, AgentDefinition, AssistantMessage, ClaudeAgentOptions, TextBlock, ToolUseBlock, query

In [7]:
load_dotenv()

True

In [8]:
from loader import load_json, load_all
claims_path="E:\\undercover\\data\\evals\\held_out_claims.json"
labels_path="E:\\undercover\\data\\evals\\held_out_decisions.json"
policy_path="E:\\undercover\\data\\policies.json"
claims=load_json(claims_path)
labels=load_json(labels_path)
policies=load_json(policy_path)

len(claims), len(labels), len(policies)

(25, 25, 50)

In [9]:
policies={policy['policy_id']: policy for policy in policies}
labels={label['claim_id']: label for label in labels}
claims={claim['claim_id']: claim for claim in claims}



In [12]:
INSTRUCTIONS=""" You ae a claims handler at Undercover Travel Insurance
working through a queue of new claims on your own.

Everything you need is in this folder:
- claims/has one file for each claim waiting in the queue
- policies/ and customers/ have the policy records and the customer records, one file each, named bu their ids
- policy_wording.md is the policy woprding and claims manual.md is the claims handling manual
- company.json has the companys limits

Decide every claim in claims/. Keep a task list with ohne task for each claim and give each claim to the handler-handler agent, which 
decides. When every claim has a decision, write handover.md: a short note for the handler on what was devided and anything they should know
"""

HANDLER_INSTUCTIONS=""" You are a claims handler at undercover travel insurance.
Decide the claim you are given, 
Read the claim in claims/, look up its policy in policies/ and its customer in customers/, and check policy_wording.md and claims_manual.md
Choose one decision:
approve: the wording covers the claim and the required documents are attached. Workout the payout using the payment section
decline: the wording does not cover the claim.
refer: a colleage needs to look at it because a required document is missing or the information disagree with the documents provided

Write the decision to decisions/<claim_id>.json with the fields claim_id, deicion, payout(in dollars) and rationale
Then reply with one line: the claim id, the decision and the payout.
"""

In [14]:
#claude agent sdk usage
TOOLS=["Read","Glob","Grep","Write","Task","TaskCreate","TaskUpdate", "TaskList","TaskGet"]

claim_handler=AgentDefinition(
description="Decides one claim and writes its decision file, Give it the claim id",
prompt=HANDLER_INSTUCTIONS,
tools=["Read", "Glob","Grep","Write"]
)

options=ClaudeAgentOptions(
    model="claude-sonnet-5-5",
    effort="low",
    cwd="E:\\FDE\\data\\DESK",
    system_prompt={"type":"preset","preset":"claude_code", "append":INSTRUCTIONS},
    tools=TOOLS,
    allowed_tools=TOOLS,
    agents={'claim-handler':claim_handler},
    permission_mode="dontAsk",
    setting_sources=[],
    env={"CLAUDE_CODE_DISABLE_AUTO_MEMORY":"1","ENABLE_CLAUDEAI_MCP_SERVERS":"false"},
    max_budget_usd=2.00
)


In [16]:
result = None

async for message in query(prompt="Please work the queue", options=options):
    if isinstance(message, AssistantMessage):
        indent = "  " if message.parent_tool_use_id else ""

        for block in message.content:
            if isinstance(block, ToolUseBlock):
                print(indent, block)
            elif isinstance(block, TextBlock):
                print(indent, block.text)

    elif isinstance(message, ResultMessage):
        result = message
        print("Result:", result)

 ToolUseBlock(id='toolu_01WNZdC7kqaDBUUKzEwtHwNp', name='Glob', input={'pattern': 'claims/*'})
 ToolUseBlock(id='toolu_01RSoQ4tomXmD6yBFKXjYwJk', name='Read', input={'file_path': 'E:\\FDE\\data\\DESK\\claims\\claims.json'})
 ToolUseBlock(id='toolu_01HKLpBdry7vKYA7azZtatEb', name='Glob', input={'pattern': '**/*'})
 ToolUseBlock(id='toolu_01RTTS6Zr7Vx7HmZdjRQPtH8', name='Read', input={'file_path': 'E:\\FDE\\data\\DESK\\claims_manual.md'})
 ToolUseBlock(id='toolu_01MMLJNv898UanRnmsboJWLx', name='Glob', input={'pattern': '.claude/agents/*'})
 ToolUseBlock(id='toolu_0114Perv639Qp1wFJZBJAkrk', name='TaskCreate', input={'subject': 'Decide CLM-1001', 'description': 'claim-handler decides CLM-1001'})
 ToolUseBlock(id='toolu_015nHdFzhKErrY3PMjQHCPsE', name='TaskCreate', input={'subject': 'Decide CLM-1002', 'description': 'claim-handler decides CLM-1002'})
 ToolUseBlock(id='toolu_01USoMU6AxGDaNTLVdUJY1gL', name='TaskCreate', input={'subject': 'Decide CLM-1003', 'description': 'claim-handler decid